# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_13 — Cointegration and Error Correction Models

### Research question

How can cointegrated price relationships be transformed into an Error Correction Model, and how can that model be used for forecasting, spread diagnostics, and relative-value research without confusing correlation with equilibrium?

This notebook follows naturally from:

```text
03_05_vector_autoregression_granger
03_10_statistical_arbitrage_pairs
03_12_kalman_filter_pairs_trading
```

The previous pairs notebooks studied static and dynamic hedge ratios. This notebook focuses on the **econometric model of cointegration**:

$$
\begin{aligned}
\Delta y_t &= \alpha \Big( y_{t-1} - \beta x_{t-1} \Big) \\
&\quad + \Gamma \Delta x_{t-1} \\
&\quad + \varepsilon_t
\end{aligned}
$$

where:

$$
y_{t-1}-\beta x_{t-1}
$$

is the error-correction term.

It covers:

1. cointegration intuition;
2. Engle-Granger two-step method;
3. residual stationarity diagnostics;
4. error correction model estimation;
5. speed-of-adjustment interpretation;
6. half-life of deviations;
7. bivariate ECM forecasting;
8. comparison with VAR in differences;
9. multivariate VECM intuition;
10. synthetic cointegrated systems;
11. train/test forecasting discipline;
12. spread trading diagnostics;
13. structural break and false cointegration warnings;
14. limitations and research extensions.

The key idea:

> Cointegration says prices can drift individually but maintain a stable long-run relationship; an error correction model says short-run returns react to deviations from that relationship.

## 1. Why cointegration matters

Two price series may both be non-stationary.

For example:

$$
x_t \sim I(1)
$$

$$
y_t \sim I(1)
$$

Their returns may be stationary, but their price levels wander.

If there exists a coefficient $\beta$ such that:

$$
s_t = y_t - \beta x_t
$$

is stationary, then $x_t$ and $y_t$ are cointegrated.

This means the two series share a long-run equilibrium relationship.

This is the econometric foundation behind many relative-value strategies:

- pairs trading;
- calendar spreads;
- spot-futures relationships;
- ETF vs basket;
- cross-listed shares;
- related commodity futures;
- yield curve relationships.

Important warning:

> Cointegration is stronger than correlation. High correlation does not imply cointegration.

## 2. Error correction model

If $x_t$ and $y_t$ are cointegrated, short-run changes can depend on the previous equilibrium error.

Define:

$$
ECT_{t-1}=y_{t-1}-\alpha-\beta x_{t-1}
$$

Then a simple bivariate ECM is:

$$
\begin{aligned}
\Delta y_t &= c_y \\
&\quad + \lambda_y ECT_{t-1} \\
&\quad + \gamma_y \Delta x_t \\
&\quad + \phi_y \Delta y_{t-1} \\
&\quad + \psi_y \Delta x_{t-1} \\
&\quad + \varepsilon^y_t
\end{aligned}
$$

and:

$$
\begin{aligned}
\Delta x_t &= c_x \\
&\quad + \lambda_x ECT_{t-1} \\
&\quad + \gamma_x \Delta y_t \\
&\quad + \phi_x \Delta x_{t-1} \\
&\quad + \psi_x \Delta y_{t-1} \\
&\quad + \varepsilon^x_t
\end{aligned}
$$

The coefficient $\lambda$ is the **speed of adjustment**.

If $ECT_{t-1}$ is high and $\lambda_y<0$, then $y$ tends to fall next period, correcting the spread.

## 3. VECM form

A Vector Error Correction Model is:

$$
\begin{aligned}
\Delta Y_t &= \Pi Y_{t-1} \\
&\quad + \sum_{i=1}^{p-1} \Gamma_i \Delta Y_{t-i} \\
&\quad + u_t
\end{aligned}
$$

where:

$$
\Pi = \alpha \beta^\top
$$

- $\beta^\top Y_{t-1}$ gives cointegration relationships;
- $\alpha$ gives speeds of adjustment.

If the cointegration rank is $r$, then there are $r$ independent long-run equilibrium relationships.

This notebook focuses primarily on the two-asset case for clarity, then shows a multivariate VECM-style extension.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from statsmodels.tsa.stattools import adfuller
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

STATSMODELS_AVAILABLE, SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class ECMConfig:
    n_obs: int = 2_500
    train_fraction: float = 0.65
    validation_fraction: float = 0.15
    annualisation: int = 252
    ecm_lags: int = 2
    z_window: int = 60
    entry_z: float = 2.0
    exit_z: float = 0.25
    cost_per_turnover: float = 0.00025
    seed: int = 42


config = ECMConfig()
config

## 5. Synthetic cointegrated and false systems

We simulate:

1. a true cointegrated pair $x,y$;
2. a structural-break pair;
3. a false correlated pair;
4. a three-variable cointegrated system.

This lets the notebook show what works and what can go wrong.

In [ ]:
def simulate_cointegration_system(config: ECMConfig) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs
    dates = pd.bdate_range("2015-01-01", periods=n)

    # Common random walk.
    dx = 0.0002 + 0.010 * rng.standard_t(df=8, size=n) * np.sqrt((8 - 2) / 8)
    x = 4.7 + np.cumsum(dx)

    # Stationary spread for true pair.
    spread = np.zeros(n)
    for t in range(1, n):
        spread[t] = 0.94 * spread[t - 1] + 0.018 * rng.standard_normal()

    alpha_true = 0.25
    beta_true = 1.12
    y = alpha_true + beta_true * x + spread

    # Structural break pair.
    spread_break = np.zeros(n)
    beta_break = np.full(n, 0.92)
    break_idx = int(0.72 * n)

    for t in range(1, n):
        if t > break_idx:
            beta_break[t] = beta_break[t - 1] + 0.00045
        else:
            beta_break[t] = beta_break[t - 1] + 0.00003 * rng.standard_normal()

        spread_break[t] = 0.95 * spread_break[t - 1] + 0.020 * rng.standard_normal()

    z_break = -0.15 + beta_break * x + spread_break

    # False correlated pair: shares x trend but no stable spread.
    false = 4.1 + 0.98 * x + np.cumsum(0.006 * rng.standard_normal(n)) + np.linspace(0, 0.45, n)

    # Three-variable system with one common stochastic trend.
    e1 = np.zeros(n)
    e2 = np.zeros(n)

    for t in range(1, n):
        e1[t] = 0.93 * e1[t - 1] + 0.015 * rng.standard_normal()
        e2[t] = 0.90 * e2[t - 1] + 0.018 * rng.standard_normal()

    a = 0.10 + 0.80 * x + e1
    b = -0.05 + 1.25 * x + e2
    c = 0.20 + 0.50 * x - 0.40 * e1 + 0.20 * e2 + 0.010 * rng.standard_normal(n)

    df = pd.DataFrame({
        "date": dates,
        "x_log": x,
        "y_log": y,
        "break_log": z_break,
        "false_log": false,
        "asset_a_log": a,
        "asset_b_log": b,
        "asset_c_log": c,
        "true_spread_xy": spread,
        "true_beta_xy": beta_true,
        "beta_break": beta_break
    })

    for col in ["x_log", "y_log", "break_log", "false_log", "asset_a_log", "asset_b_log", "asset_c_log"]:
        df[col.replace("_log", "_price")] = np.exp(df[col])

    truth = {
        "alpha_true_xy": alpha_true,
        "beta_true_xy": beta_true,
        "break_index": break_idx,
        "break_date": str(dates[break_idx])
    }

    return df, truth


data, truth = simulate_cointegration_system(config)

data.head(), truth

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["x_price", "y_price", "break_price", "false_price"]:
    plt.plot(data["date"], data[col] / data[col].iloc[0], label=col)
plt.axvline(pd.Timestamp(truth["break_date"]), linestyle="--", label="break")
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Normalised price")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(data["date"], data["true_spread_xy"])
plt.title("True Stationary Spread for x/y")
plt.xlabel("Date")
plt.ylabel("Spread")
plt.show()

## 6. Train/validation/test split

Cointegration relationships must be estimated before the test period.

We use:

1. train: estimate cointegration vector and ECM;
2. validation: sanity-check model and thresholds;
3. test: final out-of-sample evaluation.

In [ ]:
n = len(data)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train)),
    "n_validation": int(len(validation)),
    "n_test": int(len(test)),
    "train_start": str(train["date"].iloc[0]),
    "train_end": str(train["date"].iloc[-1]),
    "validation_start": str(validation["date"].iloc[0]),
    "validation_end": str(validation["date"].iloc[-1]),
    "test_start": str(test["date"].iloc[0]),
    "test_end": str(test["date"].iloc[-1])
}

pd.Series(split_metadata)

## 7. Engle-Granger step 1: estimate cointegration vector

Estimate:

$$
y_t = \alpha + \beta x_t + u_t
$$

using training data.

The residual:

$$
\hat u_t = y_t-\hat\alpha-\hat\beta x_t
$$

is the estimated equilibrium error.

In [ ]:
def estimate_cointegration_ols(df, x_col, y_col):
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)

    X = np.column_stack([np.ones(len(x)), x])
    beta_hat = np.linalg.pinv(X.T @ X) @ X.T @ y

    alpha = float(beta_hat[0])
    beta = float(beta_hat[1])
    residual = y - alpha - beta * x

    return {
        "x_col": x_col,
        "y_col": y_col,
        "alpha": alpha,
        "beta": beta,
        "residual": residual
    }


coint_xy = estimate_cointegration_ols(train, "x_log", "y_log")
coint_false = estimate_cointegration_ols(train, "x_log", "false_log")
coint_break = estimate_cointegration_ols(train, "x_log", "break_log")

pd.Series({
    "estimated_alpha_xy": coint_xy["alpha"],
    "estimated_beta_xy": coint_xy["beta"],
    "true_beta_xy": truth["beta_true_xy"],
    "estimated_beta_false": coint_false["beta"],
    "estimated_beta_break": coint_break["beta"]
})

## 8. Engle-Granger step 2: residual stationarity

If residuals are stationary, the pair may be cointegrated.

We run an ADF test when `statsmodels` is available.

Fallback diagnostics:

- mean-reversion slope;
- half-life;
- spread standard deviation.

These are not a replacement for full cointegration testing, but still useful.

In [ ]:
def mean_reversion_diagnostics(spread):
    spread = np.asarray(spread, dtype=float)
    lagged = spread[:-1]
    delta = np.diff(spread)

    X = np.column_stack([np.ones(len(lagged)), lagged])
    beta = np.linalg.pinv(X.T @ X) @ X.T @ delta

    intercept = float(beta[0])
    slope = float(beta[1])

    if slope < 0 and 1 + slope > 0:
        half_life = float(np.log(2) / (-np.log(1 + slope)))
    elif slope < 0:
        half_life = float(np.log(2) / (-slope))
    else:
        half_life = np.inf

    if STATSMODELS_AVAILABLE:
        try:
            adf = adfuller(spread, maxlag=1, regression="c", autolag=None)
            adf_stat = float(adf[0])
            adf_pvalue = float(adf[1])
        except Exception:
            adf_stat = np.nan
            adf_pvalue = np.nan
    else:
        adf_stat = np.nan
        adf_pvalue = np.nan

    return {
        "mr_intercept": intercept,
        "mr_slope": slope,
        "half_life": half_life,
        "spread_mean": float(np.mean(spread)),
        "spread_std": float(np.std(spread, ddof=1)),
        "adf_stat": adf_stat,
        "adf_pvalue": adf_pvalue
    }


coint_diag = pd.DataFrame([
    {"pair": "x_y_true", **mean_reversion_diagnostics(coint_xy["residual"])},
    {"pair": "x_break_structural", **mean_reversion_diagnostics(coint_break["residual"])},
    {"pair": "x_false_correlated", **mean_reversion_diagnostics(coint_false["residual"])}
])

coint_diag

In [ ]:
def apply_cointegration_residual(df, coint):
    return df[coint["y_col"]].to_numpy() - coint["alpha"] - coint["beta"] * df[coint["x_col"]].to_numpy()


data["spread_xy_train_beta"] = apply_cointegration_residual(data, coint_xy)
data["spread_false_train_beta"] = apply_cointegration_residual(data, coint_false)
data["spread_break_train_beta"] = apply_cointegration_residual(data, coint_break)

plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["spread_xy_train_beta"], label="true x/y spread")
plt.plot(data["date"], data["spread_false_train_beta"], label="false spread", alpha=0.8)
plt.plot(data["date"], data["spread_break_train_beta"], label="structural break spread", alpha=0.8)
plt.axvline(pd.Timestamp(truth["break_date"]), linestyle="--")
plt.axhline(0, linestyle=":")
plt.title("Estimated Cointegration Residuals Using Train Hedge Ratios")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.legend()
plt.show()

## 9. Estimate bivariate ECM

A simple ECM for $y$:

$$
\begin{aligned}
\Delta y_t &= c \\
&\quad + \lambda ECT_{t-1} \\
&\quad + \gamma_x \Delta x_{t-1} \\
&\quad + \gamma_y \Delta y_{t-1} \\
&\quad + \varepsilon_t
\end{aligned}
$$

A negative $\lambda$ means $y$ adjusts downward when it is above equilibrium.

We estimate this using training data.

In [ ]:
def build_ecm_dataset(df, coint, lags=1):
    x = df[coint["x_col"]].to_numpy(dtype=float)
    y = df[coint["y_col"]].to_numpy(dtype=float)

    ect = y - coint["alpha"] - coint["beta"] * x
    dx = np.diff(x)
    dy = np.diff(y)

    rows = []

    for t in range(lags, len(dx)):
        row = {
            "date": df["date"].iloc[t + 1],
            "dy": dy[t],
            "dx": dx[t],
            "ect_lag1": ect[t],
        }

        for lag in range(1, lags + 1):
            row[f"dy_lag{lag}"] = dy[t - lag]
            row[f"dx_lag{lag}"] = dx[t - lag]

        rows.append(row)

    return pd.DataFrame(rows)


def fit_ecm_ols(ecm_df, target="dy"):
    feature_cols = ["ect_lag1"] + [c for c in ecm_df.columns if c.startswith("dy_lag") or c.startswith("dx_lag")]
    X = np.column_stack([np.ones(len(ecm_df)), ecm_df[feature_cols].to_numpy(dtype=float)])
    y = ecm_df[target].to_numpy(dtype=float)

    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    fitted = X @ beta
    residual = y - fitted

    return {
        "feature_cols": ["intercept"] + feature_cols,
        "beta": beta,
        "fitted": fitted,
        "residual": residual,
        "sigma": float(np.std(residual, ddof=1))
    }


ecm_train_xy = build_ecm_dataset(train, coint_xy, lags=config.ecm_lags)
ecm_model_y = fit_ecm_ols(ecm_train_xy, target="dy")

pd.DataFrame({
    "term": ecm_model_y["feature_cols"],
    "coefficient": ecm_model_y["beta"]
})

## 10. Estimate ECM for both variables

A bivariate ECM can estimate both equations:

$$
\Delta y_t = f_y(ECT_{t-1}, \Delta x_{t-1}, \Delta y_{t-1})
$$

$$
\Delta x_t = f_x(ECT_{t-1}, \Delta x_{t-1}, \Delta y_{t-1})
$$

The adjustment speeds can reveal which asset does more of the correcting.

If $y$ adjusts but $x$ does not, then $y$ may be the follower.

In [ ]:
def fit_bivariate_ecm(df, coint, lags=1):
    ecm_df = build_ecm_dataset(df, coint, lags=lags)
    model_y = fit_ecm_ols(ecm_df, target="dy")
    model_x = fit_ecm_ols(ecm_df, target="dx")

    return ecm_df, model_y, model_x


ecm_train_xy, ecm_y_xy, ecm_x_xy = fit_bivariate_ecm(train, coint_xy, lags=config.ecm_lags)

ecm_coeffs = pd.DataFrame({
    "term": ecm_y_xy["feature_cols"],
    "dy_equation": ecm_y_xy["beta"],
    "dx_equation": ecm_x_xy["beta"]
})

ecm_coeffs

## 11. ECM interpretation

The error-correction coefficient is attached to:

$$
ECT_{t-1}
$$

For the $y$ equation:

- $\lambda_y < 0$: if $y$ is above equilibrium, $y$ tends to fall;
- $\lambda_y > 0$: if $y$ is above equilibrium, $y$ tends to rise further.

For the $x$ equation:

- $\lambda_x > 0$: if $y-\beta x$ is high, $x$ tends to rise, helping close the spread;
- $\lambda_x < 0$: $x$ moves away from correction.

The signs jointly determine how the spread mean-reverts.

In [ ]:
speed_report = pd.Series({
    "lambda_y_error_correction": ecm_y_xy["beta"][ecm_y_xy["feature_cols"].index("ect_lag1")],
    "lambda_x_error_correction": ecm_x_xy["beta"][ecm_x_xy["feature_cols"].index("ect_lag1")],
    "ecm_y_residual_sigma": ecm_y_xy["sigma"],
    "ecm_x_residual_sigma": ecm_x_xy["sigma"]
})

speed_report

## 12. ECM one-step forecasting

We use the fitted ECM to forecast:

$$
\Delta y_t
$$

and:

$$
\Delta x_t
$$

out of sample.

The forecast for the next log price is:

$$
\hat y_t = y_{t-1} + \widehat{\Delta y_t}
$$

This is compared with:

1. random walk baseline;
2. AR in differences;
3. ECM.

In [ ]:
def predict_ecm_on_df(df, coint, model_y, model_x, lags=1):
    ecm_df = build_ecm_dataset(df, coint, lags=lags)

    def pred(model):
        feature_cols = model["feature_cols"][1:]
        X = np.column_stack([np.ones(len(ecm_df)), ecm_df[feature_cols].to_numpy(dtype=float)])
        return X @ model["beta"]

    out = ecm_df[["date", "dy", "dx", "ect_lag1"]].copy()
    out["forecast_dy_ecm"] = pred(model_y)
    out["forecast_dx_ecm"] = pred(model_x)
    out["forecast_dy_random_walk"] = 0.0
    out["forecast_dx_random_walk"] = 0.0

    # Simple AR baseline using lagged own differences.
    out["forecast_dy_ar1"] = ecm_df["dy_lag1"]
    out["forecast_dx_ar1"] = ecm_df["dx_lag1"]

    return out


forecast_train = predict_ecm_on_df(train, coint_xy, ecm_y_xy, ecm_x_xy, lags=config.ecm_lags)
forecast_validation = predict_ecm_on_df(validation, coint_xy, ecm_y_xy, ecm_x_xy, lags=config.ecm_lags)
forecast_test = predict_ecm_on_df(test, coint_xy, ecm_y_xy, ecm_x_xy, lags=config.ecm_lags)

forecast_test.head()

In [ ]:
def forecast_metrics(forecast_df, target, pred_cols):
    rows = []

    y = forecast_df[target].to_numpy(dtype=float)

    for col in pred_cols:
        pred = forecast_df[col].to_numpy(dtype=float)
        err = pred - y

        rows.append({
            "target": target,
            "forecast": col,
            "n": len(y),
            "mse": float(np.mean(err**2)),
            "mae": float(np.mean(np.abs(err))),
            "corr": float(np.corrcoef(pred, y)[0, 1]) if np.std(pred) > 0 and np.std(y) > 0 else np.nan,
            "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(y)))
        })

    return pd.DataFrame(rows)


forecast_metrics_test = pd.concat([
    forecast_metrics(forecast_test, "dy", ["forecast_dy_ecm", "forecast_dy_random_walk", "forecast_dy_ar1"]),
    forecast_metrics(forecast_test, "dx", ["forecast_dx_ecm", "forecast_dx_random_walk", "forecast_dx_ar1"])
], ignore_index=True)

forecast_metrics_test.sort_values(["target", "mse"])

## 13. Visual forecast comparison

One-step return forecasts are noisy.

We plot predicted and realised $\Delta y$ over a sample window.

In [ ]:
sample_forecast = forecast_test.iloc[:250]

plt.figure(figsize=(12, 6))
plt.plot(sample_forecast["date"], sample_forecast["dy"], label="actual Δy", alpha=0.65)
plt.plot(sample_forecast["date"], sample_forecast["forecast_dy_ecm"], label="ECM forecast", alpha=0.85)
plt.plot(sample_forecast["date"], sample_forecast["forecast_dy_ar1"], label="AR1 baseline", alpha=0.75)
plt.axhline(0, linestyle="--")
plt.title("One-Step Δy Forecasts, Test Sample")
plt.xlabel("Date")
plt.ylabel("Δy")
plt.legend()
plt.show()

## 14. Error-correction trading signal

The ECM gives a natural spread signal:

$$
ECT_t = y_t-\alpha-\beta x_t
$$

If the spread is high, short the spread.

If the spread is low, long the spread.

We use rolling z-score:

$$
z_t=\frac{ECT_t-\mu_{t,w}}{\sigma_{t,w}}
$$

Trading rule:

- $z_t>entry$: short spread;
- $z_t<-entry$: long spread;
- $|z_t|<exit$: flat.

This is a relative-value research diagnostic, not a production backtest.

In [ ]:
def rolling_zscore(series, window):
    s = pd.Series(series)
    mean = s.rolling(window, min_periods=max(20, window // 3)).mean()
    std = s.rolling(window, min_periods=max(20, window // 3)).std()
    return ((s - mean) / std.replace(0, np.nan)).to_numpy()


def generate_positions(z, entry, exit):
    pos = np.zeros(len(z))
    current = 0.0

    for t in range(len(z)):
        zt = z[t]

        if not np.isfinite(zt):
            pos[t] = current
            continue

        if current == 0:
            if zt > entry:
                current = -1.0
            elif zt < -entry:
                current = 1.0
        else:
            if abs(zt) < exit:
                current = 0.0

        pos[t] = current

    return pos


data["ect_xy"] = apply_cointegration_residual(data, coint_xy)
data["ect_z_xy"] = rolling_zscore(data["ect_xy"], config.z_window)
data["position_xy"] = generate_positions(data["ect_z_xy"], config.entry_z, config.exit_z)

data[["date", "ect_xy", "ect_z_xy", "position_xy"]].head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["ect_z_xy"])
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axhline(0, linestyle=":")
plt.title("ECM Spread z-score")
plt.xlabel("Date")
plt.ylabel("z-score")
plt.show()

## 15. Spread strategy backtest

Spread PnL proxy:

$$
PnL_{t+1}=position_t(ECT_{t+1}-ECT_t)
$$

If long spread and spread rises, PnL is positive.

Transaction-cost proxy:

$$
cost_t=c|\Delta position_t|(1+|\beta|)
$$

This is simplified spread accounting.

In [ ]:
def backtest_spread_strategy(df, spread_col, position_col, beta, config, period_name, start_idx, end_idx):
    sub = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

    spread = sub[spread_col].to_numpy()
    position = sub[position_col].to_numpy()

    spread_change_next = np.r_[np.diff(spread), 0.0]
    gross_pnl = position * spread_change_next

    turnover = np.abs(np.diff(np.r_[0.0, position]))
    cost = config.cost_per_turnover * turnover * (1 + abs(beta))
    net_pnl = gross_pnl - cost

    out = pd.DataFrame({
        "date": sub["date"],
        "period": period_name,
        "spread": spread,
        "zscore": sub["ect_z_xy"],
        "position": position,
        "spread_change_next": spread_change_next,
        "gross_pnl": gross_pnl,
        "turnover": turnover,
        "cost": cost,
        "net_pnl": net_pnl
    })

    out["cum_net_pnl"] = out["net_pnl"].cumsum()

    return out


periods = [
    ("train", 0, train_end),
    ("validation", train_end, validation_end),
    ("test", validation_end, len(data))
]

strategy_frames = []

for period_name, start, end in periods:
    strategy_frames.append(
        backtest_spread_strategy(data, "ect_xy", "position_xy", coint_xy["beta"], config, period_name, start, end)
    )

strategy_bt = pd.concat(strategy_frames, ignore_index=True)

strategy_bt.head()

In [ ]:
def performance_summary(bt, config):
    rows = []

    for period, g in bt.groupby("period"):
        pnl = g["net_pnl"].to_numpy()
        cum = np.cumsum(pnl)
        dd = cum - np.maximum.accumulate(cum)
        active = g["position"].abs() > 0

        rows.append({
            "period": period,
            "n_days": len(g),
            "total_net_pnl": float(np.sum(pnl)),
            "annualised_pnl_proxy": float(np.mean(pnl) * config.annualisation),
            "annualised_vol_proxy": float(np.std(pnl, ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(np.mean(pnl) / np.std(pnl, ddof=1) * np.sqrt(config.annualisation)) if np.std(pnl, ddof=1) > 0 else np.nan,
            "max_drawdown_proxy": float(np.min(dd)),
            "mean_turnover": float(g["turnover"].mean()),
            "total_turnover": float(g["turnover"].sum()),
            "active_fraction": float(active.mean()),
            "hit_rate_active": float((g.loc[active, "net_pnl"] > 0).mean()) if active.any() else np.nan,
            "total_cost": float(g["cost"].sum())
        })

    return pd.DataFrame(rows)


strategy_perf = performance_summary(strategy_bt, config)

strategy_perf

In [ ]:
test_bt = strategy_bt[strategy_bt["period"] == "test"]

plt.figure(figsize=(12, 6))
plt.plot(test_bt["date"], test_bt["cum_net_pnl"])
plt.axhline(0, linestyle="--")
plt.title("Test ECM Spread Strategy PnL Proxy")
plt.xlabel("Date")
plt.ylabel("Cumulative net PnL")
plt.show()

## 16. False cointegration strategy diagnostic

We apply the same workflow to the false correlated pair.

This is a critical warning:

> If the residual is not truly stationary, z-score spread trading can lose money for structural reasons.

In [ ]:
def prepare_and_backtest_pair(data, train_df, x_col, y_col, spread_name, config):
    coint = estimate_cointegration_ols(train_df, x_col, y_col)
    data_tmp = data.copy()
    data_tmp[spread_name] = apply_cointegration_residual(data_tmp, coint)
    data_tmp[f"{spread_name}_z"] = rolling_zscore(data_tmp[spread_name], config.z_window)
    data_tmp[f"position_{spread_name}"] = generate_positions(data_tmp[f"{spread_name}_z"], config.entry_z, config.exit_z)

    frames = []
    for period_name, start, end in periods:
        bt = backtest_spread_strategy(
            data_tmp.rename(columns={f"{spread_name}_z": "ect_z_xy"}),
            spread_name,
            f"position_{spread_name}",
            coint["beta"],
            config,
            period_name,
            start,
            end
        )
        bt["pair"] = f"{x_col}_{y_col}"
        frames.append(bt)

    return coint, pd.concat(frames, ignore_index=True)


false_coint, false_bt = prepare_and_backtest_pair(
    data,
    train,
    "x_log",
    "false_log",
    "ect_false",
    config
)

break_coint, break_bt = prepare_and_backtest_pair(
    data,
    train,
    "x_log",
    "break_log",
    "ect_break",
    config
)

false_perf = performance_summary(false_bt, config)
break_perf = performance_summary(break_bt, config)

pd.concat([
    false_perf.assign(pair="x_false"),
    break_perf.assign(pair="x_break")
], ignore_index=True)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(false_bt[false_bt["period"] == "test"]["date"], false_bt[false_bt["period"] == "test"]["cum_net_pnl"], label="false pair")
plt.plot(break_bt[break_bt["period"] == "test"]["date"], break_bt[break_bt["period"] == "test"]["cum_net_pnl"], label="break pair")
plt.axhline(0, linestyle="--")
plt.title("False / Broken Pair Test PnL")
plt.xlabel("Date")
plt.ylabel("Cumulative net PnL")
plt.legend()
plt.show()

## 17. Cost sensitivity

Mean-reversion strategies often have turnover.

We vary transaction costs and test whether the ECM spread strategy survives.

In [ ]:
def run_cost_sensitivity(config, cost_grid):
    rows = []

    for cost in cost_grid:
        cfg = ECMConfig(**{**asdict(config), "cost_per_turnover": cost})
        frames = []

        for period_name, start, end in periods:
            frames.append(
                backtest_spread_strategy(data, "ect_xy", "position_xy", coint_xy["beta"], cfg, period_name, start, end)
            )

        bt = pd.concat(frames, ignore_index=True)
        perf = performance_summary(bt, cfg)
        perf["cost_per_turnover"] = cost
        rows.append(perf)

    return pd.concat(rows, ignore_index=True)


cost_grid = [0.0, 0.00010, 0.00025, 0.00050, 0.00100, 0.00200]
cost_sensitivity = run_cost_sensitivity(config, cost_grid)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for period, g in cost_sensitivity.groupby("period"):
    plt.plot(g["cost_per_turnover"], g["sharpe_proxy"], marker="o", label=period)
plt.axhline(0, linestyle="--")
plt.title("ECM Strategy Cost Sensitivity")
plt.xlabel("Cost per turnover")
plt.ylabel("Sharpe proxy")
plt.legend()
plt.show()

## 18. Parameter sensitivity

Entry threshold choice can be overfit.

We test several entry z-scores on validation and test.

In [ ]:
def run_threshold_sensitivity(config, entry_grid):
    rows = []

    for entry in entry_grid:
        cfg = ECMConfig(**{**asdict(config), "entry_z": entry})
        data_tmp = data.copy()
        data_tmp["position_tmp"] = generate_positions(data_tmp["ect_z_xy"], entry, cfg.exit_z)

        frames = []
        for period_name, start, end in periods:
            frames.append(
                backtest_spread_strategy(data_tmp, "ect_xy", "position_tmp", coint_xy["beta"], cfg, period_name, start, end)
            )

        bt = pd.concat(frames, ignore_index=True)
        perf = performance_summary(bt, cfg)
        perf["entry_z"] = entry
        rows.append(perf)

    return pd.concat(rows, ignore_index=True)


entry_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
threshold_sensitivity = run_threshold_sensitivity(config, entry_grid)

threshold_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for period, g in threshold_sensitivity.groupby("period"):
    plt.plot(g["entry_z"], g["sharpe_proxy"], marker="o", label=period)
plt.axhline(0, linestyle="--")
plt.title("ECM Strategy Threshold Sensitivity")
plt.xlabel("Entry z-score")
plt.ylabel("Sharpe proxy")
plt.legend()
plt.show()

## 19. Structural break diagnostics

Cointegration can break.

We check rolling hedge ratio and rolling residual stationarity diagnostics.

A relationship that is cointegrated in train can fail later.

In [ ]:
def rolling_cointegration_diagnostics(df, x_col, y_col, window=252, step=21):
    rows = []

    for start in range(0, len(df) - window + 1, step):
        sub = df.iloc[start:start + window]
        coint = estimate_cointegration_ols(sub, x_col, y_col)
        diag = mean_reversion_diagnostics(coint["residual"])

        rows.append({
            "start_date": sub["date"].iloc[0],
            "end_date": sub["date"].iloc[-1],
            "alpha": coint["alpha"],
            "beta": coint["beta"],
            **diag
        })

    return pd.DataFrame(rows)


rolling_xy = rolling_cointegration_diagnostics(data, "x_log", "y_log", window=252, step=21)
rolling_break = rolling_cointegration_diagnostics(data, "x_log", "break_log", window=252, step=21)
rolling_false = rolling_cointegration_diagnostics(data, "x_log", "false_log", window=252, step=21)

rolling_xy.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_xy["end_date"], rolling_xy["beta"], label="true x/y")
plt.plot(rolling_break["end_date"], rolling_break["beta"], label="break pair")
plt.plot(rolling_false["end_date"], rolling_false["beta"], label="false pair")
plt.axvline(pd.Timestamp(truth["break_date"]), linestyle="--")
plt.title("Rolling Cointegration Hedge Ratio")
plt.xlabel("Window end")
plt.ylabel("Beta")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(rolling_xy["end_date"], rolling_xy["half_life"], label="true x/y")
plt.plot(rolling_break["end_date"], rolling_break["half_life"], label="break pair")
plt.plot(rolling_false["end_date"], rolling_false["half_life"], label="false pair")
plt.axhline(60, linestyle="--")
plt.title("Rolling Half-Life Diagnostics")
plt.xlabel("Window end")
plt.ylabel("Half-life")
plt.legend()
plt.show()

## 20. Multivariate VECM-style example

For three variables:

$$
Y_t =
\begin{bmatrix}
a_t\\
b_t\\
c_t
\end{bmatrix}
$$

a VECM is:

$$
\begin{aligned}
\Delta Y_t &= \alpha \beta^\top Y_{t-1} \\
&\quad + \Gamma \Delta Y_{t-1} \\
&\quad + u_t
\end{aligned}
$$

Here we demonstrate a simplified version:

1. estimate cointegration residuals by regressing $b$ and $c$ on $a$;
2. use both residuals as error-correction terms;
3. forecast $\Delta a,\Delta b,\Delta c$.

This is not a Johansen implementation, but it illustrates multivariate error correction.

In [ ]:
def estimate_multivariate_equilibrium(train_df):
    # Use asset_a_log as common trend anchor.
    coint_b = estimate_cointegration_ols(train_df, "asset_a_log", "asset_b_log")
    coint_c = estimate_cointegration_ols(train_df, "asset_a_log", "asset_c_log")
    return coint_b, coint_c


def build_multivariate_ecm_dataset(df, coint_b, coint_c, lags=1):
    a = df["asset_a_log"].to_numpy()
    b = df["asset_b_log"].to_numpy()
    c = df["asset_c_log"].to_numpy()

    ect_b = b - coint_b["alpha"] - coint_b["beta"] * a
    ect_c = c - coint_c["alpha"] - coint_c["beta"] * a

    da = np.diff(a)
    db = np.diff(b)
    dc = np.diff(c)

    rows = []

    for t in range(lags, len(da)):
        row = {
            "date": df["date"].iloc[t + 1],
            "da": da[t],
            "db": db[t],
            "dc": dc[t],
            "ect_b_lag1": ect_b[t],
            "ect_c_lag1": ect_c[t],
            "da_lag1": da[t - 1],
            "db_lag1": db[t - 1],
            "dc_lag1": dc[t - 1]
        }
        rows.append(row)

    return pd.DataFrame(rows)


def fit_multivariate_ecm(train_df):
    coint_b, coint_c = estimate_multivariate_equilibrium(train_df)
    mv = build_multivariate_ecm_dataset(train_df, coint_b, coint_c, lags=1)

    feature_cols = ["ect_b_lag1", "ect_c_lag1", "da_lag1", "db_lag1", "dc_lag1"]
    X = np.column_stack([np.ones(len(mv)), mv[feature_cols].to_numpy()])

    models = {}
    for target in ["da", "db", "dc"]:
        y = mv[target].to_numpy()
        beta = np.linalg.pinv(X.T @ X) @ X.T @ y
        models[target] = beta

    return {
        "coint_b": coint_b,
        "coint_c": coint_c,
        "feature_cols": ["intercept"] + feature_cols,
        "models": models,
        "train_dataset": mv
    }


mv_model = fit_multivariate_ecm(train)

pd.DataFrame({
    "term": mv_model["feature_cols"],
    "da_eq": mv_model["models"]["da"],
    "db_eq": mv_model["models"]["db"],
    "dc_eq": mv_model["models"]["dc"]
})

In [ ]:
def predict_multivariate_ecm(df, mv_model):
    mv = build_multivariate_ecm_dataset(df, mv_model["coint_b"], mv_model["coint_c"], lags=1)
    feature_cols = mv_model["feature_cols"][1:]
    X = np.column_stack([np.ones(len(mv)), mv[feature_cols].to_numpy()])

    out = mv[["date", "da", "db", "dc"]].copy()

    for target in ["da", "db", "dc"]:
        out[f"forecast_{target}_mvecm"] = X @ mv_model["models"][target]
        out[f"forecast_{target}_zero"] = 0.0

    return out


mv_test_forecasts = predict_multivariate_ecm(test, mv_model)

mv_metrics = pd.concat([
    forecast_metrics(mv_test_forecasts, "da", ["forecast_da_mvecm", "forecast_da_zero"]),
    forecast_metrics(mv_test_forecasts, "db", ["forecast_db_mvecm", "forecast_db_zero"]),
    forecast_metrics(mv_test_forecasts, "dc", ["forecast_dc_mvecm", "forecast_dc_zero"])
], ignore_index=True)

mv_metrics

## 21. Practical checklist for ECM/VECM research

Before trusting an ECM:

1. **Check integration order**  
   Are price levels plausibly $I(1)$?

2. **Estimate cointegration on train only**  
   Do not fit hedge ratios using test data.

3. **Test residual stationarity**  
   A non-stationary spread invalidates the ECM idea.

4. **Interpret adjustment speeds**  
   Do signs make economic sense?

5. **Compare forecasting baselines**  
   Does ECM beat random walk or AR in differences?

6. **Validate out of sample**  
   A cointegrated relationship can break.

7. **Watch half-life**  
   Is mean reversion fast enough to trade?

8. **Test costs**  
   Relative-value strategies can be turnover-heavy.

9. **Monitor structural breaks**  
   Cointegration is not permanent.

10. **Do not confuse ECM with arbitrage**  
   Statistical equilibrium can fail.

## 22. Saving outputs

The notebook saves:

1. synthetic cointegration data;
2. split metadata;
3. cointegration diagnostics;
4. ECM coefficients;
5. forecast tables;
6. forecast metrics;
7. spread strategy backtests;
8. cost sensitivity;
9. threshold sensitivity;
10. rolling cointegration diagnostics;
11. multivariate ECM forecasts;
12. manifest.

In [ ]:
output_dir = Path("data/processed/cointegration_error_correction_model_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_cointegration_data.csv"
split_path = output_dir / "split_metadata.json"
coint_diag_path = output_dir / "cointegration_diagnostics.csv"
ecm_coeffs_path = output_dir / "ecm_coefficients.csv"
forecast_train_path = output_dir / "forecast_train.csv"
forecast_validation_path = output_dir / "forecast_validation.csv"
forecast_test_path = output_dir / "forecast_test.csv"
forecast_metrics_path = output_dir / "forecast_metrics_test.csv"
strategy_bt_path = output_dir / "strategy_backtest.csv"
strategy_perf_path = output_dir / "strategy_performance.csv"
false_bt_path = output_dir / "false_pair_backtest.csv"
break_bt_path = output_dir / "break_pair_backtest.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
threshold_sensitivity_path = output_dir / "threshold_sensitivity.csv"
rolling_xy_path = output_dir / "rolling_cointegration_xy.csv"
rolling_break_path = output_dir / "rolling_cointegration_break.csv"
rolling_false_path = output_dir / "rolling_cointegration_false.csv"
mv_metrics_path = output_dir / "multivariate_ecm_metrics.csv"
mv_forecasts_path = output_dir / "multivariate_ecm_test_forecasts.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
Path(split_path).write_text(json.dumps(split_metadata, indent=2, default=str))
coint_diag.to_csv(coint_diag_path, index=False)
ecm_coeffs.to_csv(ecm_coeffs_path, index=False)
forecast_train.to_csv(forecast_train_path, index=False)
forecast_validation.to_csv(forecast_validation_path, index=False)
forecast_test.to_csv(forecast_test_path, index=False)
forecast_metrics_test.to_csv(forecast_metrics_path, index=False)
strategy_bt.to_csv(strategy_bt_path, index=False)
strategy_perf.to_csv(strategy_perf_path, index=False)
false_bt.to_csv(false_bt_path, index=False)
break_bt.to_csv(break_bt_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
threshold_sensitivity.to_csv(threshold_sensitivity_path, index=False)
rolling_xy.to_csv(rolling_xy_path, index=False)
rolling_break.to_csv(rolling_break_path, index=False)
rolling_false.to_csv(rolling_false_path, index=False)
mv_metrics.to_csv(mv_metrics_path, index=False)
mv_test_forecasts.to_csv(mv_forecasts_path, index=False)

manifest = {
    "dataset_name": "cointegration_error_correction_model_outputs",
    "schema_version": "cointegration_error_correction_model_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "truth": truth,
    "statsmodels_available": STATSMODELS_AVAILABLE,
    "scipy_available": SCIPY_AVAILABLE,
    "cointegration_xy": {
        "alpha": coint_xy["alpha"],
        "beta": coint_xy["beta"]
    },
    "ecm_speed_report": speed_report.to_dict(),
    "strategy_performance": strategy_perf.to_dict(orient="records"),
    "forecast_metrics_test": forecast_metrics_test.to_dict(orient="records"),
    "notes": [
        "Synthetic data includes true cointegrated pair, false correlated pair, structural break pair, and multivariate cointegrated system.",
        "Cointegration vector is estimated on train data only.",
        "ECM forecasts are evaluated out of sample.",
        "Spread strategy uses simplified error-correction residual PnL and turnover cost proxy.",
        "Multivariate ECM section is illustrative and not a full Johansen VECM implementation.",
        "Notebook emphasises that cointegration can break and is not arbitrage."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, coint_diag_path, ecm_coeffs_path, strategy_perf_path, manifest_path

## 23. Limitations

### 23.1 Synthetic data

The notebook uses synthetic data.

Real price data has contract rolls, missing data, asynchronous closes, liquidity differences, and structural breaks.

### 23.2 Engle-Granger simplicity

The two-step Engle-Granger approach is simple but limited.

It depends on the choice of dependent variable and handles only one cointegration relationship.

### 23.3 ADF test approximation

ADF p-values for generated residuals require special Engle-Granger critical values.

This notebook reports ADF diagnostics but does not claim formal inference precision.

### 23.4 Simplified multivariate ECM

The multivariate section is illustrative, not a full Johansen VECM.

A production VECM should estimate rank and cointegration vectors jointly.

### 23.5 Linear adjustment

The ECM assumes linear error correction.

Real spreads may have threshold effects, nonlinear adjustment, and regime dependence.

### 23.6 Simplified trading PnL

The spread strategy is a proxy.

Real execution requires two-leg fills, slippage, contract multipliers, margin, borrow/funding, and liquidity constraints.

### 23.7 Cointegration can break

A pair can be cointegrated historically and fail out of sample.

## 24. Research frontier and extensions

Important extensions include:

1. **Johansen VECM**  
   Estimate multivariate cointegration rank and vectors jointly.

2. **Threshold ECM**  
   Allow adjustment only when deviations are large.

3. **Regime-switching ECM**  
   Different adjustment speeds in different regimes.

4. **Kalman-filter cointegration**  
   Dynamic cointegration vector and time-varying equilibrium.

5. **Bayesian VECM**  
   Incorporate parameter uncertainty.

6. **Break detection**  
   Detect failed cointegration relationships.

7. **Nonlinear error correction**  
   Use kernels, trees, or neural networks for adjustment.

8. **Cointegration baskets**  
   Trade multi-asset spreads rather than pairs.

9. **Execution-aware relative value**  
   Incorporate costs, liquidity, and legging risk.

10. **Chinese futures calendar and cross-commodity spreads**  
   Apply ECM to main/deferred contracts and related commodity chains.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_14_information_coefficient_analysis**  
   Evaluate spread and ECM signals as factors.

2. **04_04_dynamic_covariance_and_correlation**  
   Model changing relationships among assets.

3. **05_01_vectorized_backtest_engine**  
   Generalise spread strategy backtesting.

4. **05_03_transaction_costs_slippage_latency**  
   Add realistic two-leg execution costs.

5. **07_01_multi_asset_cta_research_pipeline**  
   Combine trend, carry, volatility, and relative-value signals.

6. **Chinese futures relative-value capstone**  
   ECM for calendar spreads, commodity chains, and night-session dynamics.

## 26. Summary

This notebook implemented cointegration and Error Correction Models.

It showed how to:

1. simulate cointegrated and false correlated systems;
2. estimate a cointegration vector with Engle-Granger OLS;
3. test residual mean reversion and stationarity;
4. estimate bivariate ECM equations;
5. interpret speed-of-adjustment coefficients;
6. forecast short-run changes out of sample;
7. compare ECM with random walk and AR baselines;
8. build an error-correction spread signal;
9. backtest a simplified spread strategy;
10. diagnose false and broken pairs;
11. test cost and threshold sensitivity;
12. monitor rolling cointegration stability;
13. demonstrate a simplified multivariate ECM.

The key computational takeaway:

> ECMs combine long-run equilibrium information with short-run dynamics in a leakage-controlled forecasting model.

The key financial takeaway:

> Cointegration can support relative-value research, but only if the equilibrium relationship is stable, economically plausible, and tradable after costs.

## 27. Further reading

- Engle and Granger, "Co-integration and Error Correction: Representation, Estimation, and Testing."
- Johansen, *Likelihood-Based Inference in Cointegrated Vector Autoregressive Models.*
- Hamilton, *Time Series Analysis.*
- Lütkepohl, *New Introduction to Multiple Time Series Analysis.*
- Vidyamurthy, *Pairs Trading.*
- Literature on threshold cointegration, regime-switching ECM, and cointegration-based statistical arbitrage.